# Unsloth Fine-Tuning for Colab

Trains Llama-3.2-3B-Instruct on Hindi instruction dataset (all 200k samples)

## Setup Instructions
1. Upload your `Hindi.jsonl` file to Colab (or mount Google Drive)
2. Update the dataset path in Step 1 if needed
3. **Mount Google Drive** in Step 0.5 to save checkpoints (recommended for free tier)
4. Run all cells sequentially

## Checkpointing
- Checkpoints are saved every ~1 hour (500 steps)
- Training automatically resumes from the latest checkpoint if found
- Checkpoints are saved to Google Drive for persistence across sessions


In [ ]:
# Install unsloth (run this once per Colab session)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset, concatenate_datasets
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import os
import json
from collections import Counter, defaultdict
import random
import glob

print("=" * 70)
print("Unsloth Fine-Tuning - Colab")
print("=" * 70)


## Step 0.5: Setup Checkpoint Directory (Google Drive)

💾 **Important**: Mount Google Drive to save checkpoints that persist across Colab sessions!


In [ ]:
# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

# Setup checkpoint directory DIRECTLY on Google Drive (for persistence)
checkpoint_dir = "/content/drive/MyDrive/unsloth_checkpoints"

# Create directory if it doesn't exist
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"✓ Checkpoint directory (Drive): {checkpoint_dir}")
print("\n💡 Checkpoints will be saved every ~1 hour directly to Google Drive")


## Step 1: Load Dataset


In [ ]:
print("\n[Step 1] Loading Hindi dataset...")

# Option 1: If you uploaded the file to Colab, use this path:
# hindi_path = "/content/Hindi.jsonl"

# Option 2: If using Google Drive, mount it first and use:
# from google.colab import drive
# drive.mount('/content/drive')
# hindi_path = "/content/drive/MyDrive/path/to/Hindi.jsonl"

# Option 3: If the file is in the same directory as this notebook:
# For Colab, you can upload files using the file browser or use:
# from google.colab import files
# uploaded = files.upload()  # Then use the uploaded filename

# Default: Try to find the file (adjust path as needed)
hindi_path = "/content/Hindi.jsonl"  # Change this to your file path

# Check if dataset exists
if not os.path.exists(hindi_path):
    raise FileNotFoundError(f"Hindi dataset not found: {hindi_path}\nPlease upload the file or update the path above.")

print(f"  Hindi dataset: {hindi_path}")

# Load dataset
dataset = load_dataset("json", data_files=hindi_path, split="train")

print(f"  ✓ Loaded {len(dataset):,} Hindi samples")
print(f"  ✓ Using all {len(dataset):,} samples for training (no sampling)")

# Show first example
if len(dataset) > 0:
    print("\n  First example:")
    print(json.dumps(dataset[0], ensure_ascii=False, indent=2)[:500])


## Step 2: Load Model


In [ ]:
print("\n[Step 2] Loading model...")

max_seq_length = 2048
dtype = None  # Auto-detect
load_in_4bit = True

print("  Loading Llama-3.2-3B-Instruct model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("  ✓ Model loaded successfully")


## Step 3: Format Dataset


In [ ]:
print("\n[Step 3] Formatting dataset...")

# Multilingual prompt template
multilingual_prompt = """You are a helpful multilingual assistant capable of understanding and responding in multiple Indian languages including Hindi, Tamil, Telugu, Kannada, Malayalam, Bengali, Gujarati, Marathi, Punjabi, Odia, Assamese, Urdu, and more.

Please respond to the following in the same language as the input:

{}
"""
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    if 'instruction' in examples and 'output' in examples:
        for instruction, output in zip(examples['instruction'], examples['output']):
            text = multilingual_prompt.format(f"{instruction}\n\n{output}") + EOS_TOKEN
            texts.append(text)
    elif 'text' in examples:
        for text in examples['text']:
            texts.append(text + EOS_TOKEN)
    else:
        raise ValueError(f"Unknown dataset format. Keys: {list(examples.keys())}")
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print("  ✓ Dataset formatted successfully")


## Step 4: Configure LoRA


In [ ]:
print("\n[Step 4] Configuring LoRA...")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
print("  ✓ LoRA configuration applied")


## Step 5: Setup Trainer


In [ ]:
print("\n[Step 5] Setting up trainer...")

# Calculate steps per hour (roughly 1000 steps = ~1 hour on T4 GPU)
# Adjust save_steps based on your GPU speed:
# - T4: ~1000 steps/hour
# - V100: ~1500 steps/hour  
# - A100: ~2500 steps/hour
save_steps = 1000  # Save checkpoint every ~1 hour

# Check for existing checkpoints on Google Drive
def find_latest_checkpoint(checkpoint_dir):
    """Find the latest checkpoint in the directory (on Drive)"""
    checkpoints = glob.glob(os.path.join(checkpoint_dir, "checkpoint-*"))
    if not checkpoints:
        return None
    latest = max(checkpoints, key=os.path.getctime)
    return latest

resume_from_checkpoint = find_latest_checkpoint(checkpoint_dir)

if resume_from_checkpoint:
    print(f"  ✓ Found existing checkpoint: {resume_from_checkpoint}")
    print(f"  Training will resume from step {os.path.basename(resume_from_checkpoint).split('-')[1]}")
else:
    print("  No existing checkpoint found. Starting fresh training.")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=100,
        max_steps=25000,  # 1 full epoch for ~200k samples
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=checkpoint_dir,  # Save checkpoints here
        report_to="none",
        # Checkpointing settings
        save_strategy="steps",
        save_steps=save_steps,  # Save every ~1 hour
        save_total_limit=3,  # Keep only last 3 checkpoints to save space
        load_best_model_at_end=False,
        # Resume from checkpoint
        resume_from_checkpoint=resume_from_checkpoint if resume_from_checkpoint else None,
    ),
)
print("  ✓ Trainer configured and ready")


## (Deprecated) Manual Sync to Google Drive

You no longer need to run this. Checkpoints are now written directly to Google Drive every ~1 hour.


In [ ]:
# Deprecated: Manual sync is no longer needed because
# checkpoints are saved directly to Google Drive via `checkpoint_dir`.
pass


## Step 6: Train Model

⚠️ **Note**: This may take 6-8 hours depending on your GPU. 

### Checkpointing Info:
- Checkpoints are saved every **~1 hour** (500 steps)
- If your session disconnects, just re-run from Step 0.5 onwards
- Training will automatically resume from the latest checkpoint
- Checkpoints are synced to Google Drive for persistence


In [ ]:
print("\n" + "=" * 70)
print("[Step 6] Starting training...")
print("=" * 70)
print("This may take 6-8 hours depending on your GPU.")
print(f"Checkpoints will be saved every {save_steps} steps (~1 hour)")
print("=" * 70)

try:
    trainer.train()
    
    print("=" * 70)
    print("  ✓ Training complete!")
    print("=" * 70)

except KeyboardInterrupt:
    print("\n⚠️ Training interrupted!")
    print("Don't worry - your progress is saved in checkpoints.")
    print("Re-run from Step 0.5 to resume training.")
except Exception as e:
    print(f"\n⚠️ Error during training: {e}")
    print("Your progress is saved in checkpoints.")
    print("Re-run from Step 0.5 to resume training.")
    raise


## Step 7: Save Final Model

💾 **Save to Google Drive**: After training completes, save the final model to Google Drive.


In [ ]:
print("\n[Step 7] Saving final model...")

# Save to Google Drive (persistent)
output_dir_drive = "/content/drive/MyDrive/lora_model"
os.makedirs(output_dir_drive, exist_ok=True)

# Load the best checkpoint if available
latest_checkpoint = find_latest_checkpoint(checkpoint_dir)
if latest_checkpoint:
    print(f"  Loading from checkpoint: {latest_checkpoint}")
    # The model is already loaded from the checkpoint, just save it
    model.save_pretrained(output_dir_drive)
    tokenizer.save_pretrained(output_dir_drive)
    print(f"  ✓ Final model saved to Google Drive: {output_dir_drive}")
else:
    # Save current model state
    model.save_pretrained(output_dir_drive)
    tokenizer.save_pretrained(output_dir_drive)
    print(f"  ✓ Model saved to Google Drive: {output_dir_drive}")

# Also save locally (optional)
output_dir_local = "/content/lora_model"
model.save_pretrained(output_dir_local)
tokenizer.save_pretrained(output_dir_local)
print(f"  ✓ Model also saved locally: {output_dir_local}")

print("\n" + "=" * 70)
print("✅ All done! Model saved successfully!")
print("=" * 70)
